In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import os
import matplotlib.pyplot as plt

# Initialize the budget_entries DataFrame globally
# This DataFrame will store all budget entries
budget_entries = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Category'])

# Define the filepath for saving and loading data
budget_filepath = 'budget_data.csv'

# --- Widget Definitions ---

# Input widgets for adding new entries
date_picker = widgets.DatePicker(description='📅 Date:')
description_text = widgets.Text(description='📝 Description:')
amount_float = widgets.FloatText(description='💰 Amount:')
category_dropdown = widgets.Dropdown(
    options=['Income', 'Expense'],
    description=' 분류 Category:'
)

# Filtering widgets for displaying specific entries
filter_start_date = widgets.DatePicker(description='Start Date:')
filter_end_date = widgets.DatePicker(description='End Date:')
filter_category = widgets.Dropdown(
    options=['All', 'Income', 'Expense'],
    description='Category:'
)
filter_description = widgets.Text(description='Keyword:')

# Action buttons for various operations
add_button = widgets.Button(description='➕ Add Entry')
display_button = widgets.Button(description='📊 Display Entries')
summary_button = widgets.Button(description='📈 Show Summary')
save_button = widgets.Button(description='💾 Save Budget')
load_button = widgets.Button(description='📂 Load Budget')
close_button = widgets.Button(description='❌ Close Program')
remove_button = widgets.Button(description='🗑️ Remove Selected Entries')
apply_filter_button = widgets.Button(description='Apply Filter')
clear_filter_button = widgets.Button(description='Clear Filter')
visualize_button = widgets.Button(description='📊 Visualize Budget')

# Output widget to display results, messages, and plots
output = widgets.Output()

# --- Layout Definitions ---

# Group input widgets visually
input_widgets = widgets.VBox([
    widgets.Label(value="## 📥 Input Details"), # Chapter heading for input section
    date_picker,
    description_text,
    amount_float,
    category_dropdown
])

# Group filtering widgets visually
filter_widgets = widgets.VBox([
    widgets.Label(value="## 🔍 Filter Entries"), # Chapter heading for filter section
    filter_start_date,
    filter_end_date,
    filter_category,
    filter_description
])

# Group action buttons horizontally
action_buttons = widgets.HBox([
    widgets.Label(value="## ⚙️ Actions"), # Chapter heading for action buttons
    add_button,
    display_button,
    summary_button,
    visualize_button,
    save_button,
    load_button,
    close_button,
    remove_button,
    apply_filter_button,
    clear_filter_button
])

# Combine all sections into the main application layout
main_layout = widgets.VBox([
    widgets.Label(value="# 💰 Personal Budget Tracker"), # Main application heading
    input_widgets,
    filter_widgets,
    action_buttons,
    widgets.Label(value="## 📊 Output"), # Chapter heading for output area
    output
])

# Dictionary to store checkboxes generated when displaying entries.
# Key: original DataFrame index, Value: Checkbox widget
selection_checkboxes = {}

# Variable to hold the currently filtered DataFrame for display purposes.
# Set to None when no filter is active.
filtered_budget_entries = None

# --- Function Definitions ---

def add_budget_entry(b):
    """Adds a new budget entry based on widget values.

    Args:
        b (ipywidgets.Button): The button widget that triggered the event.
    """
    global budget_entries, filtered_budget_entries
    with output:
        clear_output() # Clear previous output
        try:
            # Get values from input widgets
            date_value = date_picker.value
            description_value = description_text.value
            amount_value = amount_float.value
            category_value = category_dropdown.value

            # Validate input values (basic check)
            if not date_value or not description_value or amount_value is None:
                 print("❗ Please fill in Date, Description, and Amount.")
                 return

            if amount_value < 0:
                 print("❗ Amount cannot be negative.")
                 return

            # Create a new entry as a DataFrame row
            new_entry_data = {
                'Date': [date_value],
                'Description': [description_value],
                'Amount': [amount_value],
                'Category': [category_value]
            }
            new_entry_df = pd.DataFrame(new_entry_data)

            # Append the new entry to the main budget DataFrame
            # Use ignore_index=True to re-index the DataFrame
            budget_entries = pd.concat([budget_entries, new_entry_df], ignore_index=True)

            # Clear input widgets after successful addition
            date_picker.value = None
            description_text.value = ''
            amount_float.value = 0.0
            category_dropdown.value = 'Income' # Reset to default category

            # Provide success feedback to the user
            print("✅ Budget entry added successfully!")

            # Reset filtered view as the underlying data has changed
            filtered_budget_entries = None

        except (ValueError, TypeError) as e:
            # Handle specific input errors (though ipywidgets handles some)
            print(f"Error adding entry: Invalid input for Amount or Date. Details: {e}")
        except Exception as e:
            # Handle any other unexpected errors
            print(f"An unexpected error occurred while adding entry: {e}")


def display_entries(b):
    """Displays the budget entries in a table format with checkboxes for selection,
       applying the current filter if active.

    Args:
        b (ipywidgets.Button): The button widget that triggered the event.
    """
    global selection_checkboxes, filtered_budget_entries
    with output:
        clear_output() # Clear previous output

        # Determine which DataFrame to display: filtered or the full one
        df_to_display = filtered_budget_entries if filtered_budget_entries is not None else budget_entries

        if df_to_display.empty:
            # Message for empty state
            print("No entries yet or no entries match the current filter criteria.")
        else:
            # Create a copy to avoid modifying the original DataFrame when adding display elements
            display_df = df_to_display.copy()

            # Clear previous checkboxes from the dictionary
            selection_checkboxes = {}
            checkbox_widgets = []

            # Add a checkbox for each entry in the displayed DataFrame
            for index in display_df.index:
                checkbox = widgets.Checkbox(value=False, description='')
                # Store checkbox linked to the ORIGINAL index of the entry in budget_entries
                # This is important for removal to work correctly on the original DataFrame
                # Find the original index by matching the row data
                # This approach assumes rows are unique. A unique ID column would be more robust.
                try:
                    # Find the index in the original budget_entries that matches the current row
                    original_index_iloc = budget_entries.index[budget_entries.eq(display_df.loc[index]).all(axis=1)]
                    if not original_index_iloc.empty:
                         original_index = original_index_iloc[0]
                         selection_checkboxes[original_index] = checkbox
                         # Display the original index next to the checkbox for user reference
                         checkbox_widgets.append(widgets.HBox([widgets.Label(value=str(original_index)), checkbox]))
                    else:
                         # Handle cases where a row in filtered_df might not be found in budget_entries
                         # (e.g., due to complex manipulations, though unlikely with current functions)
                         print(f"Warning: Could not find original index for displayed row {index}. Checkbox not added.")

                except Exception as e:
                     # Catch any other unexpected errors during checkbox creation
                     print(f"Error creating checkbox for row {index}: {e}")


            # Display the section header and the checkboxes
            print("## Select Entries to Remove:")
            display(widgets.VBox(checkbox_widgets))

            # Display the DataFrame content with formatted date
            if 'Date' in display_df.columns and not display_df['Date'].empty:
                 # Use style.format for cleaner date display in the notebook output
                 display(display_df.style.format({'Date': lambda x: x.strftime('%Y-%m-%d') if pd.notnull(x) else ''}))
            else:
                 # Display without date formatting if Date column is missing or empty
                 display(display_df)


def calculate_summary(b):
    """Calculates and displays the budget summary (total income, expenses, and balance).

    Args:
        b (ipywidgets.Button): The button widget that triggered the event.
    """
    with output:
        clear_output() # Clear previous output
        if budget_entries.empty:
            print("No entries yet to calculate summary.")
            return

        try:
            # Ensure 'Amount' column is numeric for calculation
            budget_entries['Amount'] = pd.to_numeric(budget_entries['Amount'], errors='coerce').fillna(0)

            # Separate income and expense entries based on 'Category'
            income_entries = budget_entries[budget_entries['Category'] == 'Income']
            expense_entries = budget_entries[budget_entries['Category'] == 'Expense']

            # Calculate totals for income and expenses
            total_income = income_entries['Amount'].sum()
            total_expenses = expense_entries['Amount'].sum()

            # Calculate the balance
            balance = total_income - total_expenses

            # Display the summary results with formatting
            print("--- 📈 Budget Summary ---")
            print(f"Total Income: £{total_income:.2f}")
            print(f"Total Expenses: £{total_expenses:.2f}")
            print(f"Balance: £{balance:.2f}")
            print("--------------------")

        except KeyError as e:
             # Handle cases where expected columns are missing
             print(f"Error calculating summary: Missing expected column '{e}'. Ensure 'Amount' and 'Category' columns exist.")
        except Exception as e:
             # Handle any other unexpected errors during calculation
             print(f"An unexpected error occurred while calculating summary: {e}")


def save_budget(filepath=budget_filepath):
    """Saves the budget data to a CSV file.

    Args:
        filepath (str): The path to the CSV file. Defaults to 'budget_data.csv'.
    """
    with output:
        clear_output() # Clear previous output
        try:
            # Save the DataFrame to CSV, without the pandas index
            budget_entries.to_csv(filepath, index=False)
            print(f"💾 Budget data saved successfully to {filepath}")
        except IOError as e:
            # Handle file writing errors (e.g., permission issues, disk full)
            print(f"Error saving budget data: Could not write to file {filepath}. Details: {e}")
        except Exception as e:
             # Handle any other unexpected errors during saving
             print(f"An unexpected error occurred while saving budget data: {e}")


def load_budget(filepath=budget_filepath):
    """Loads the budget data from a CSV file.

    Args:
        filepath (str): The path to the CSV file. Defaults to 'budget_data.csv'.
    """
    global budget_entries, filtered_budget_entries
    with output:
        clear_output() # Clear previous output
        try:
            # Check if the file exists before attempting to read
            if os.path.exists(filepath):
                # Read the CSV file into the budget_entries DataFrame
                budget_entries = pd.read_csv(filepath)
                print(f"📂 Budget data loaded successfully from {filepath}")

                # Convert 'Date' column to datetime objects for proper handling (filtering, visualization)
                if 'Date' in budget_entries.columns and not budget_entries['Date'].empty:
                    # Use errors='coerce' to turn any invalid date strings into NaT (Not a Time)
                    original_dtype = budget_entries['Date'].dtype
                    budget_entries['Date'] = pd.to_datetime(budget_entries['Date'], errors='coerce')
                    # Check if any values became NaT after conversion
                    if budget_entries['Date'].isnull().any():
                         print("Warning: Some dates in the loaded file were invalid and set to 'Not a Time'.")
                    # If the original dtype wasn't datetime and no errors, confirm successful conversion
                    elif not pd.api.types.is_datetime64_any_dtype(original_dtype):
                         print("Date column converted to datetime objects.")

                # Ensure 'Amount' column is numeric after loading
                if 'Amount' in budget_entries.columns:
                    budget_entries['Amount'] = pd.to_numeric(budget_entries['Amount'], errors='coerce').fillna(0)
                    if budget_entries['Amount'].isnull().any():
                         print("Warning: Some amounts in the loaded file were invalid and set to 0.")
                else:
                    print("Warning: 'Amount' column not found in loaded data. Summary/Visualization may not work correctly.")

                # Ensure 'Category' column exists and has expected values (optional but good practice)
                if 'Category' not in budget_entries.columns:
                    print("Warning: 'Category' column not found in loaded data. Summary/Filtering may not work correctly.")
                    # Add a default category column if missing
                    budget_entries['Category'] = 'Expense' # Or some other default

            else:
                # If the file does not exist, initialize an empty DataFrame
                budget_entries = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Category'])
                print(f"No budget data file found at {filepath}. Initialized an empty budget.")

            # Reset filtered view after loading new data
            filtered_budget_entries = None

        except FileNotFoundError:
             # Explicitly catch FileNotFoundError (already handled by os.path.exists, but adds clarity)
             budget_entries = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Category'])
             filtered_budget_entries = None
             print(f"No budget data file found at {filepath}. Initialized an empty budget.")
        except pd.errors.EmptyDataError:
             # Handle case where file exists but is empty
             budget_entries = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Category'])
             filtered_budget_entries = None
             print(f"Budget data file at {filepath} is empty. Initialized an empty budget.")
        except pd.errors.ParserError as e:
             # Handle issues with parsing the CSV file format
             budget_entries = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Category'])
             filtered_budget_entries = None
             print(f"Error parsing budget data file: {e}. Initialized an empty budget.")
        except Exception as e:
            # Handle any other unexpected errors during loading
            budget_entries = pd.DataFrame(columns=['Date', 'Description', 'Amount', 'Category'])
            filtered_budget_entries = None
            print(f"Error loading budget data: {e}. Initialized an empty budget.")


def remove_selected_entries(b):
    """Removes the selected budget entries based on the checkboxes displayed.

    Args:
        b (ipywidgets.Button): The button widget that triggered the event.
    """
    global budget_entries, filtered_budget_entries
    # Collect the original indices of entries that have their checkbox selected
    indices_to_remove = [index for index, checkbox in selection_checkboxes.items() if checkbox.value]

    with output:
        clear_output() # Clear previous output
        if not indices_to_remove:
            # Message if no entries were selected
            print("No entries selected for removal.")
            return

        try:
            # Remove entries from the main budget DataFrame using their original indices
            # Use errors='ignore' to prevent error if an index to remove is not found
            original_entry_count = len(budget_entries)
            budget_entries = budget_entries.drop(indices_to_remove, errors='ignore').reset_index(drop=True)
            removed_count = original_entry_count - len(budget_entries)


            # Reset filtered view after modifying the main data
            filtered_budget_entries = None

            # Provide feedback on how many entries were removed
            print(f"🗑️ Removed {removed_count} selected entries.")

            # Automatically refresh the display to show the updated list
            display_entries(None)

        except KeyError as e:
            # Handle cases where a specified index doesn't exist (though errors='ignore' should prevent this)
            print(f"Error removing entries: Invalid index found. Details: {e}")
        except Exception as e:
            # Handle any other unexpected errors during removal
            print(f"An unexpected error occurred while removing entries: {e}")


def filter_entries(b):
    """Filters the budget entries based on the values in the filter widgets.

    Args:
        b (ipywidgets.Button): The button widget that triggered the event.
    """
    global filtered_budget_entries
    with output:
        clear_output() # Clear previous output
        try:
            # Start with a copy of the main budget data to apply filters
            filtered_df = budget_entries.copy()

            # Ensure 'Date' column is datetime for date filtering
            if 'Date' in filtered_df.columns:
                 filtered_df['Date'] = pd.to_datetime(filtered_df['Date'], errors='coerce')
                 # Filter out rows with invalid dates (NaT) if date filtering is applied
                 if filter_start_date.value or filter_end_date.value:
                     filtered_df.dropna(subset=['Date'], inplace=True)
                     if filtered_df.empty and (filter_start_date.value or filter_end_date.value):
                          print("No valid date entries match the date filter.")
                          filtered_budget_entries = filtered_df # Set to empty filtered_df
                          display_entries(None)
                          return # Exit if no data left after dropping NaT and applying date filter

            else:
                # Message if Date column is missing but date filter is attempted
                if filter_start_date.value or filter_end_date.value:
                     print("Warning: 'Date' column not found. Cannot apply date filters.")


            # Apply date range filter if dates are provided
            start_date = filter_start_date.value
            end_date = filter_end_date.value
            if start_date:
                # Convert start_date widget value to datetime for comparison
                start_dt = pd.to_datetime(start_date)
                filtered_df = filtered_df[filtered_df['Date'] >= start_dt]
            if end_date:
                # Convert end_date widget value to datetime for comparison
                end_dt = pd.to_datetime(end_date)
                filtered_df = filtered_df[filtered_df['Date'] <= end_dt]

            # Apply category filter if a specific category is selected
            category = filter_category.value
            if category != 'All':
                # Ensure 'Category' column exists before filtering
                if 'Category' in filtered_df.columns:
                    filtered_df = filtered_df[filtered_df['Category'] == category]
                else:
                    print("Warning: 'Category' column not found. Cannot apply category filter.")


            # Apply description keyword filter if a keyword is entered
            keyword = filter_description.value.strip()
            if keyword:
                # Ensure 'Description' column exists and is string type before searching
                if 'Description' in filtered_df.columns:
                    # Fill potential NaNs in 'Description' with empty string before searching
                    filtered_df = filtered_df[filtered_df['Description'].astype(str).str.contains(keyword, case=False, na=False)]
                else:
                    print("Warning: 'Description' column not found. Cannot apply keyword filter.")


            # Store the resulting filtered DataFrame
            filtered_budget_entries = filtered_df

            # Display the filtered entries
            display_entries(None)

        except (ValueError, TypeError) as e:
             # Handle errors related to invalid input values or types in filters
             print(f"Error applying filter: Invalid input or data type. Details: {e}")
        except KeyError as e:
             # Handle cases where required columns are missing during filtering
             print(f"Error applying filter: Missing expected column '{e}'. Ensure 'Date', 'Category', and 'Description' columns exist.")
        except Exception as e:
             # Handle any other unexpected errors during filtering
             print(f"An unexpected error occurred while applying filter: {e}")


def clear_filters(b):
    """Clears the applied filters and displays all budget entries.

    Args:
        b (ipywidgets.Button): The button widget that triggered the event.
    """
    global filtered_budget_entries
    # Reset the filtered DataFrame to None
    filtered_budget_entries = None

    # Clear the values in the filter widgets
    filter_start_date.value = None
    filter_end_date.value = None
    filter_category.value = 'All' # Reset category to default 'All'
    filter_description.value = ''

    with output:
        clear_output() # Clear previous output
        print("✅ Filters cleared. Displaying all entries.")

    # Display all entries (by calling display_entries with filtered_budget_entries set to None)
    display_entries(None)


def visualize_budget(b):
    """Generates and displays budget visualizations (monthly trends and category distribution)."""
    with output:
        clear_output() # Clear previous output
        if budget_entries.empty:
            # Message for empty state
            print("No entries yet to visualize.")
            return

        try:
            # Ensure 'Date' column exists and is datetime for time-based visualization
            if 'Date' in budget_entries.columns:
                # Convert 'Date' column to datetime, coercing errors to NaT
                df_viz = budget_entries.copy() # Work on a copy for visualization
                df_viz['Date'] = pd.to_datetime(df_viz['Date'], errors='coerce')
                # Drop rows with invalid dates (NaT) for time series analysis
                df_viz.dropna(subset=['Date', 'Amount', 'Category'], inplace=True)

                if df_viz.empty:
                    print("No valid data (Date, Amount, Category) to generate visualizations.")
                    return

            else:
                 print("Error: 'Date' column not found for visualization.")
                 return

            # --- Monthly Income vs Expenses Trend ---

            # Aggregate data by month using resampling, summing 'Amount'
            # Filter for Income and Expense before resampling
            monthly_income = df_viz[df_viz['Category'] == 'Income'].set_index('Date').resample('M')['Amount'].sum().reset_index()
            monthly_expenses = df_viz[df_viz['Category'] == 'Expense'].set_index('Date').resample('M')['Amount'].sum().reset_index()

            # Format the date for plotting the x-axis
            if not monthly_income.empty:
                monthly_income['Month'] = monthly_income['Date'].dt.strftime('%Y-%m')
            if not monthly_expenses.empty:
                monthly_expenses['Month'] = monthly_expenses['Date'].dt.strftime('%Y-%m')

            # Create the plot
            fig1, ax1 = plt.subplots(figsize=(12, 6))
            # Plot income if data exists
            if not monthly_income.empty:
                ax1.plot(monthly_income['Month'], monthly_income['Amount'], label='Income', marker='o', color='green')
            # Plot expenses if data exists
            if not monthly_expenses.empty:
                ax1.plot(monthly_expenses['Month'], monthly_expenses['Amount'], label='Expenses', marker='o', color='red')

            ax1.set_title('Monthly Income vs Expenses')
            ax1.set_xlabel('Month')
            ax1.set_ylabel('Amount (£)')
            ax1.legend()
            ax1.tick_params(axis='x', rotation=45) # Rotate x-axis labels for readability
            ax1.grid(True, linestyle='--', alpha=0.6) # Add a grid
            plt.tight_layout() # Adjust layout to prevent labels overlapping
            plt.show(fig1) # Display the first figure

            # --- Expense Distribution by Category (Pie Chart) ---

            # Filter for only expense entries
            expense_entries = df_viz[df_viz['Category'] == 'Expense'].copy()

            if not expense_entries.empty:
                # Ensure 'Description' is string type and handle potential NaNs before grouping
                expense_entries['Description'] = expense_entries['Description'].astype(str).fillna('Unknown')
                # Group expenses by description and sum the amounts
                category_expenses = expense_entries.groupby('Description')['Amount'].sum()

                if not category_expenses.empty and category_expenses.sum() > 0: # Check if there are actual expenses
                    # Create the pie chart
                    fig2, ax2 = plt.subplots(figsize=(8, 8))
                    ax2.pie(category_expenses, labels=category_expenses.index, autopct='%1.1f%%', startangle=90, colors=plt.cm.Paired.colors)
                    ax2.set_title('Expense Distribution by Description') # Title changed to description for clarity
                    ax2.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
                    plt.tight_layout()
                    plt.show(fig2) # Display the second figure
                elif category_expenses.empty:
                     print("No detailed expense descriptions to display for the pie chart.")
                else: # category_expenses not empty, but sum is 0
                     print("Total expenses are zero, cannot generate pie chart.")
            else:
                print("No expense entries to display for the pie chart.")

        except KeyError as e:
             # Handle cases where required columns are missing
             print(f"Error visualizing budget: Missing expected column '{e}'. Ensure 'Date', 'Amount', and 'Category' columns exist.")
        except Exception as e:
             # Handle any other unexpected errors during visualization
             print(f"An unexpected error occurred while visualizing budget: {e}")


# --- Event Handling ---

# Link buttons to their corresponding functions
add_button.on_click(add_budget_entry)
display_button.on_click(display_entries)
summary_button.on_click(calculate_summary)
save_button.on_click(lambda b: save_budget()) # Use lambda for functions without the button argument
load_button.on_click(lambda b: load_budget())
close_button.on_click(lambda b: save_budget()) # Save data when 'Close Program' is clicked
remove_button.on_click(remove_selected_entries)
apply_filter_button.on_click(filter_entries)
clear_filter_button.on_click(clear_filters)
visualize_button.on_click(visualize_budget)

# --- Initial Load ---

# Automatically load budget data when the notebook cell is run
load_budget()

# Display the main layout to show the interactive widgets
display(main_layout)